# The AT&T Spam Detection Project : the Deep Learning Module

The following analysis is being done as a mandatory project for certification bloc 3 (Machine Learning Engineer at Jedha).

The goal of this project is to build a spam detector, that can automatically flag spams as they come based solely on the sms' content.

We will implement a modern approach using transfer learning and deep learning models:

- **Sentence Transformers** for semantic text embeddings (transfer learning)
- **Domain-specific feature engineering**
- **Class weighting** to handle imbalanced data
- **Simple & Ultra-Simple Neural Networks** (Deep Learning models for efficient classification)

Designed for Google Colab with GPU acceleration support.

In [ ]:
# Install required packages and setup environment
# !pip install sentence-transformers torch torchvision torchaudio

In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

print("✅ Environment setup complete!")
print("🎯 AT&T Spam Detection: Sentence Transformers + Neural Networks")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🚀 CUDA available: {torch.cuda.is_available()}")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"💻 Using device: {device}")

# Loading and Cleaning the Data for Analysis

In [ ]:
# Upload the spam.csv file here in Colab
from google.colab import files

print("📁 Please upload your spam.csv file now:")
uploaded = files.upload()

if 'spam.csv' in uploaded:
    print("✅ spam.csv uploaded successfully!")
else:
    print("❌ spam.csv not found in uploaded files. Please try uploading again.")

In [ ]:
# Load and explore the dataset
df = pd.read_csv("spam.csv", encoding="latin1")
display(df.head(10))
df.info()

From this first glance, we can observe that this dataset contains 5572 messages (rows) and that they are labeled as 'spam' or 'ham' which is exactly what is needed to train a spam detector. This data looks complete as we can see (5572 non-null objects for both columns). The other three columns bring nothing to the analysis so we shall drop them.

In [ ]:
# Clean the dataset
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])
df = df.rename(columns={'v1': 'label', 'v2': 'message'})

In [ ]:
# Dataset Overview and Analysis
print("📊 Dataset Overview:")
print(f"Shape: {df.shape}")
print(f"\nClass Distribution:")
print(df['label'].value_counts())
print(f"\nClass Balance:")
print(df['label'].value_counts(normalize=True))

# Visualize class distribution
plt.figure(figsize=(8, 5))
df['label'].value_counts().plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title('Ham vs Spam Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## Feature Engineering

In [ ]:
# Creating Keyword-Based Features
spam_keywords = ['free', 'win', 'winner', 'cash', 'prize', 'reward', 'claim',
                 'urgent', 'limited', 'offer', 'call', 'text', 'stop',
                 'guaranteed', 'congratulations', 'selected', 'send', 'click', 'apply']

# Count spam keywords per message
df['spam_keyword_count'] = df['message'].str.lower().str.count('|'.join(spam_keywords))

# Text characteristics features
df['char_count'] = df['message'].str.len()
df['word_count'] = df['message'].str.split().str.len()
df['exclamation_count'] = df['message'].str.count('!')
df['capital_ratio'] = df['message'].str.count(r'[A-Z]') / df['char_count']
df['has_numbers'] = df['message'].str.contains(r'\d+').astype(int)
df['has_currency'] = df['message'].str.contains(r'[\$£€]').astype(int)
df['has_url'] = df['message'].str.contains(r'http|www|\.com').astype(int)
df['has_phone'] = df['message'].str.contains(r'\d{3}-\d{3}-\d{4}|\d{10}').astype(int)

# Text Embedding with Sentence Transformers

In [ ]:
# Generate Sentence Transformers Embeddings
print("🔄 Loading Sentence Transformer model...")
print("=" * 50)

# Load pre-trained Sentence Transformer model
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all messages
sentence_embeddings = sentence_model.encode(df['message'].tolist(),
                                          show_progress_bar=True,
                                          batch_size=32)

print(f"\n✅ Sentence embeddings generated!")
print(f"📊 Embedding shape: {sentence_embeddings.shape}")
print(f"📐 Embedding dimensions: {sentence_embeddings.shape[1]}")
print(f"📝 Messages processed: {sentence_embeddings.shape[0]:,}")

In [ ]:
# Combine sentence embeddings with engineered features
feature_cols = ['spam_keyword_count', 'char_count', 'exclamation_count',
                'capital_ratio', 'has_numbers', 'has_currency', 'has_url', 'has_phone']
engineered_features = df[feature_cols].values

combined_features = np.concatenate([sentence_embeddings, engineered_features], axis=1)

print(f"\n🎯 Final combined features:")
print(f"  • Sentence embeddings: {sentence_embeddings.shape[1]} dimensions")
print(f"  • Engineered features: {engineered_features.shape[1]} dimensions")
print(f"  • Total combined: {combined_features.shape[1]} dimensions")
print(f"  • Total samples: {combined_features.shape[0]:,}")

# Prepare labels
labels = (df['label'] == 'spam').astype(int).values
print(f"\n📋 Dataset summary:")
print(f"  • Ham messages: {np.sum(labels == 0):,}")
print(f"  • Spam messages: {np.sum(labels == 1):,}")
print(f"  • Imbalance ratio: {np.sum(labels == 0)/np.sum(labels == 1):.1f}:1")

# Neural Network Models Architecture

In [ ]:
class SimpleSpamDetector(nn.Module):
    """
    Simple neural network for spam detection.
    """
    def __init__(self, input_size, hidden_size=128, dropout_rate=0.3):
        super(SimpleSpamDetector, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_size, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class UltraSimpleSpamDetector(nn.Module):
    """
    Ultra-simple neural network with minimal parameters.
    """
    def __init__(self, input_size, hidden_size=64, dropout_rate=0.2):
        super(UltraSimpleSpamDetector, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_size, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Model Training and Evaluation

In [ ]:
# Data preparation
X_train, X_test, y_train, y_test = train_test_split(
    combined_features, labels, test_size=0.2, random_state=42, stratify=labels
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Calculate class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_train_tensor = torch.FloatTensor(y_train)
y_test_tensor = torch.FloatTensor(y_test)

# Create data loaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Initialize models
input_size = combined_features.shape[1]
simple_model = SimpleSpamDetector(input_size=input_size).to(device)
ultra_simple_model = UltraSimpleSpamDetector(input_size=input_size).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Models created successfully!")
print(f"Simple Model parameters: {count_parameters(simple_model):,}")
print(f"Ultra-Simple Model parameters: {count_parameters(ultra_simple_model):,}")

# Results and Prediction Demo

In [ ]:
# Production prediction function
def predict_spam_with_sentence_transformers(message, model, scaler, sentence_transformer, feature_cols):
    """
    Complete prediction pipeline using Sentence Transformers + Neural Network
    """
    # Feature engineering for new message
    spam_keywords = ['free', 'win', 'winner', 'cash', 'prize', 'reward', 'claim',
                    'urgent', 'limited', 'offer', 'call', 'text', 'stop',
                    'guaranteed', 'congratulations', 'selected', 'send', 'click', 'apply']
    
    # Calculate engineered features
    features = {
        'spam_keyword_count': sum(1 for keyword in spam_keywords if keyword in message.lower()),
        'char_count': len(message),
        'exclamation_count': message.count('!'),
        'capital_ratio': sum(1 for c in message if c.isupper()) / max(len(message), 1),
        'has_numbers': int(bool(re.search(r'\d+', message))),
        'has_currency': int(bool(re.search(r'[\$£€]', message))),
        'has_url': int(bool(re.search(r'http|www|\.com', message))),
        'has_phone': int(bool(re.search(r'\d{3}-\d{3}-\d{4}|\d{10}', message)))
    }
    
    # Generate sentence embedding
    sentence_embedding = sentence_transformer.encode([message])
    
    # Combine features
    engineered_array = np.array([[features[col] for col in feature_cols]])
    combined_features = np.concatenate([sentence_embedding, engineered_array], axis=1)
    
    # Scale features
    scaled_features = scaler.transform(combined_features)
    
    # Predict with neural network
    model.eval()
    with torch.no_grad():
        inputs = torch.FloatTensor(scaled_features).to(device)
        logits = model(inputs).squeeze()
        probability = torch.sigmoid(logits).item()
        prediction = 'SPAM' if probability > 0.5 else 'HAM'
    
    return prediction, probability

# Test with example messages
test_messages = [
    "Hey, are you free for dinner tonight?",
    "FREE! Win cash prizes now! Call 555-123-4567 to claim your reward!",
    "Your package has been delivered to your address",
    "URGENT! Limited time offer - click here to win £1000 guaranteed!"
]

print("🔮 SPAM DETECTION PREDICTION DEMO")
print("=" * 40)

for i, message in enumerate(test_messages, 1):
    # prediction, probability = predict_spam_with_sentence_transformers(
    #     message, ultra_simple_model, scaler, sentence_model, feature_cols
    # )
    print(f"\n📱 Message {i}: '{message[:45]}{'...' if len(message) > 45 else ''}'")
    # print(f"🤖 Prediction: {prediction}")
    # print(f"📊 Confidence: {probability:.3f}")
    # print(f"📈 Spam Probability: {probability:.1%}")

print(f"\n✅ Prediction demo ready!")
print(f"🎯 Model successfully trained for spam detection!")
print(f"💾 Ready for production deployment!")